# 1. Data Collection — SpaceX API
This notebook obtains historical launch records from the public SpaceX API and saves a reproducible raw snapshot. The API schema can evolve; inspect the fields before using them downstream.

In [ ]:
import requests
import pandas as pd
from pathlib import Path
from datetime import datetime, timezone

ROOT = Path("..")
RAW = ROOT / "data" / "raw"
RAW.mkdir(parents=True, exist_ok=True)

url = "https://api.spacexdata.com/v4/launches/query"
payload = {
    "query": {},
    "options": {
        "select": {
            "name": 1, "date_utc": 1, "success": 1, "flight_number": 1,
            "rocket": 1, "payloads": 1, "launchpad": 1, "cores": 1,
            "failures": 1, "details": 1
        },
        "pagination": False
    }
}
response = requests.post(url, json=payload, timeout=60)
response.raise_for_status()
launches = response.json().get("docs", [])
print("Retrieved launch records:", len(launches))
print("Retrieved at UTC:", datetime.now(timezone.utc).isoformat())
pd.json_normalize(launches).head()

In [ ]:
raw_path = RAW / "spacex_launches_api.json"
raw_path.write_text(response.text, encoding="utf-8")
df_api = pd.json_normalize(launches)
df_api.to_csv(RAW / "spacex_launches_api.csv", index=False)
print("Saved:", raw_path, "and CSV snapshot")

## Notes
This notebook retrieves all launch records returned by the API query. Later notebooks must filter to Falcon 9 and define the first-stage landing target carefully. A mission-level `success` field is not the same as a booster landing outcome.